In [1]:
import os
import numpy as np
import tensorflow as tf
import cv2
import matplotlib.pyplot as plt

from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import EfficientNetB4
from tensorflow.keras.applications.efficientnet import preprocess_input
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

from sklearn.utils.class_weight import compute_class_weight

In [2]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [13]:
DATA_PATH =  "/content/drive/MyDrive/data"

In [16]:
IMG_SIZE = (300,300)
BATCH_SIZE = 32
# Ensure DATA_PATH is defined, e.g., DATA_PATH = './dataset'
# 1. Training Generator (WITH data augmentation)
ImageDataGenerator(
    rescale=1./255,
    rotation_range=30,
    zoom_range=0.2,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True,
    vertical_flip=True,   # VERY important for satellite
    brightness_range=[0.8,1.2]
)
# 2. Validation Generator (NO augmentation, only preprocessing)
val_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    validation_split=0.2
)
# 3. Create flow from directory
train_gen = train_datagen.flow_from_directory(
    DATA_PATH,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='training',
    seed=42  # Use same seed to ensure splits don't overlap
)
val_gen = val_datagen.flow_from_directory(
    DATA_PATH,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='validation',
    shuffle=False, # Usually False for validation to keep labels in order
    seed=42        # Must match the training seed
)

Found 3133 images belonging to 4 classes.
Found 781 images belonging to 4 classes.


In [17]:
class_weights = {
    0: 1.5,   # damage
    1: 1.4,   # flood
    2: 1.0,   # normal
    3: 0.8    # wildfire
}

In [9]:
print(train_gen.num_classes)
print(train_gen.class_indices)

4
{'damage': 0, 'flood': 1, 'normal': 2, 'wildfire': 3}


In [18]:
base_model = EfficientNetB4(
    weights='imagenet',
    include_top=False,
    input_shape=(300,300,3)
)

# Freeze layers
for layer in base_model.layers:
    layer.trainable = False

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(256, activation='relu')(x)
x = Dropout(0.5)(x)

output = Dense(train_gen.num_classes, activation='softmax')(x)

model = Model(inputs=base_model.input, outputs=output)

In [19]:
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

callbacks = [
    EarlyStopping(
        monitor='val_loss',
        patience=5,
        restore_best_weights=True
    ),
    ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.3,
    patience=3,
    min_lr=1e-6
)
]

In [20]:
# Unfreeze last layers
for layer in base_model.layers[-40:]:
    layer.trainable = True

# Add your new loss and compilation code here:
loss = tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1)

model.compile(
    optimizer=Adam(learning_rate=1e-5),
    loss=loss,
    metrics=['accuracy', tf.keras.metrics.Precision(), tf.keras.metrics.Recall()]
)

# Keep the original model.fit below it
history_finetune = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=35,
    class_weight=class_weights,
    callbacks=callbacks
)


Epoch 1/35
98/98 ━━━━━━━━━━━━━━━━━━━━ 1074s 10s/step - accuracy: 0.4360 - loss: 1.3608 - precision: 0.6467 - recall: 0.0380 - val_accuracy: 0.4379 - val_loss: 1.3077 - val_precision: 0.0000e+00 - val_recall: 0.0000e+00 - learning_rate: 1.0000e-05
Epoch 2/35
98/98 ━━━━━━━━━━━━━━━━━━━━ 148s 2s/step - accuracy: 0.7092 - loss: 1.1118 - precision: 0.9015 - recall: 0.2132 - val_accuracy: 0.6223 - val_loss: 1.1955 - val_precision: 0.8537 - val_recall: 0.0896 - learning_rate: 1.0000e-05
Epoch 3/35
98/98 ━━━━━━━━━━━━━━━━━━━━ 150s 2s/step - accuracy: 0.8299 - loss: 0.9236 - precision: 0.9406 - recall: 0.5056 - val_accuracy: 0.6581 - val_loss: 1.1208 - val_precision: 0.8578 - val_recall: 0.2548 - learning_rate: 1.0000e-05
Epoch 4/35
98/98 ━━━━━━━━━━━━━━━━━━━━ 152s 2s/step - accuracy: 0.8573 - loss: 0.8087 - precision: 0.9482 - recall: 0.6712 - val_accuracy: 0.6645 - val_loss: 1.0683 - val_precision: 0.8678 - val_recall: 0.3867 - learning_rate: 1.0000e-05
Epoch 5/35
98/98 ━━━━━━━━━━━━━━━━━━━━ 152s

In [21]:
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np

y_true = val_gen.classes
y_pred = model.predict(val_gen)
y_pred_classes = np.argmax(y_pred, axis=1)

print(classification_report(y_true, y_pred_classes))

25/25 ━━━━━━━━━━━━━━━━━━━━ 39s 924ms/step
              precision    recall  f1-score   support

           0       0.72      0.98      0.83        98
           1       0.86      0.96      0.90       112
           2       0.89      0.51      0.65       293
           3       0.70      0.90      0.79       278

    accuracy                           0.77       781
   macro avg       0.79      0.84      0.79       781
weighted avg       0.80      0.77      0.76       781



In [ ]:
from sklearn.metrics import accuracy_score
acc=accuracy_score()

In [ ]:
model.save("model.h5")

In [ ]:
model.save("../app/model.h5")

In [ ]:
#GRADCAM
def get_gradcam(model, img_array, layer_name="top_conv"):
    grad_model = tf.keras.models.Model(
        [model.inputs],
        [model.get_layer(layer_name).output, model.output]
    )

    with tf.GradientTape() as tape:
        conv_outputs, predictions = grad_model(img_array)
        class_idx = tf.argmax(predictions[0])
        loss = predictions[:, class_idx]

    grads = tape.gradient(loss, conv_outputs)
    pooled_grads = tf.reduce_mean(grads, axis=(0,1,2))

    conv_outputs = conv_outputs[0]
    heatmap = conv_outputs @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)

    heatmap = np.maximum(heatmap, 0) / np.max(heatmap)
    return heatmap.numpy()

In [ ]:
def show_gradcam(img_path):
    img = cv2.imread(img_path)
    img = cv2.resize(img, (224,224))

    img_array = np.expand_dims(img, axis=0)
    img_array = preprocess_input(img_array)

    heatmap = get_gradcam(model, img_array)

    heatmap = cv2.resize(heatmap, (224,224))
    heatmap = np.uint8(255 * heatmap)
    heatmap = cv2.applyColorMap(heatmap, cv2.COLORMAP_JET)

    superimposed = heatmap * 0.4 + img

    plt.imshow(cv2.cvtColor(superimposed.astype("uint8"), cv2.COLOR_BGR2RGB))
    plt.axis("off")
    plt.show()

In [ ]:
show_gradcam("test.jpg")